In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from utils import load_set
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_regression, RFE
from scipy.stats import pearsonr
import pickle

In [3]:
X_train, y_train = load_set('data/df_train_cleaned.csv')

In [4]:
def feature_selection_summary(X: pd.DataFrame, y: pd.Series, n: int = 100):
    """
    Dla podanej ramki danych X i zmiennej docelowej y:
    - wykonuje test korelacji Pearsona,
    - wykonuje test mutual information,
    - przeprowadza RFE z regresją liniową,
    - trenuje Random Forest i pobiera feature importance.
    Zwraca listę unikalnych cech zsumowanych po wszystkich metodach.
    """
    features = X.columns
    
    # 1️⃣ Test korelacji Pearsona
    pearson_scores = []
    for col in features:
        try:
            corr, _ = pearsonr(X[col], y)
            pearson_scores.append(abs(corr))
        except Exception:
            pearson_scores.append(0)
    pearson_top = list(X.columns[np.argsort(pearson_scores)[-n:]])
    
    # 2️⃣ Test mutual information
    mi = mutual_info_regression(X, y, random_state=42)
    mi_top = list(X.columns[np.argsort(mi)[-n:]])
    
    # 3️⃣ RFE z regresją liniową
    model = LogisticRegression(max_iter=1000)
    rfe = RFE(model, n_features_to_select=n)
    rfe.fit(X, y)
    rfe_top = list(X.columns[rfe.support_])
    
    # 4️⃣ Random Forest feature importance
    rf = RandomForestClassifier(random_state=47)
    rf.fit(X, y)
    rf_importances = rf.feature_importances_
    rf_top = list(X.columns[np.argsort(rf_importances)[-n:]])
    
    # 🔁 Suma mnogościowa (union)
    all_selected = set(pearson_top) | set(mi_top) | set(rfe_top) | set(rf_top)
    
    return list(all_selected)

In [5]:
len(X_train.columns)

187

In [6]:
selected_features = feature_selection_summary(X_train, y_train, n=30)
print("Wybrane cechy:")
selected_features

Wybrane cechy:


['numeric_pipeline__-log_wsk_struktury_kapitalu',
 'numeric_pipeline__-Rozliczenia_miedzyokresowe_dlugie',
 'numeric_pipeline__wsk_stopa_zysku_sprzedaz_sqrt',
 'numeric_pipeline__-log_Kapital_zapasowy',
 'numeric_pipeline__wsk_ebitda_koszty_finansowe_1_squared',
 'numeric_pipeline__-Zobowiazania_dlugoterminowe',
 'numeric_pipeline__-podatek_dochodowy_sqrt',
 'numeric_pipeline__-wsk_obrotowosc_gotowkowa_squared',
 'numeric_pipeline__-log_wsk_zadluzenia_gotowki_1',
 'numeric_pipeline__-current_ratio',
 'numeric_pipeline__-wsk_sprzedaz_kap_obrotowy',
 'numeric_pipeline__wsk_koszty_fin_przychody',
 'numeric_pipeline__-Rozliczenia_miedzyokresowe_krotkie',
 'numeric_pipeline__-Naleznosci_dostaw_uslug_12m_pozostale',
 'numeric_pipeline__-wsk_ROA',
 'numeric_pipeline__-log_Naleznosci_dostaw_uslug_12m_powiazane',
 'numeric_pipeline__Zobowiazania_dostaw_uslug_12m_powiazane',
 'numeric_pipeline__log_wsk_zadluzenia',
 'numeric_pipeline__log_wsk_rotacja_rz_aktywow_trwalych',
 'numeric_pipeline__-lo

In [7]:
print(len(selected_features))

93


In [8]:
with open(r"models\selected_features.pkl", "wb") as f:   # 'wb' = write binary
    pickle.dump(selected_features,f)